In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Live_Query_Engine_Test") \
    .master("local[2]") \
    .config("spark.jars.packages",
            "org.apache.iceberg:iceberg-spark-runtime-4.0_2.13:1.10.1,"
            "org.apache.hadoop:hadoop-aws:3.4.1,"
            "software.amazon.awssdk:bundle:2.29.38") \
    .config("spark.sql.extensions",
            "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions") \
    .config("spark.sql.catalog.local",           "org.apache.iceberg.spark.SparkCatalog") \
    .config("spark.sql.catalog.local.type",      "rest") \
    .config("spark.sql.catalog.local.uri",       "http://rest-catalog:8181") \
    .config("spark.sql.catalog.local.warehouse", "s3a://business-data/") \
    .config("spark.sql.catalog.local.s3.endpoint",          "http://minio:9000") \
    .config("spark.sql.catalog.local.s3.path-style-access", "true") \
    .config("spark.sql.catalog.local.s3.region",            "us-east-1") \
    .config("spark.hadoop.fs.s3a.endpoint",          "http://minio:9000") \
    .config("spark.hadoop.fs.s3a.access.key",        "admin") \
    .config("spark.hadoop.fs.s3a.secret.key",        "password") \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .getOrCreate()

print(f"Spark Session Ready! Version: {spark.version}")

Spark Session Ready! Version: 4.0.2


In [2]:
# Query the live Iceberg table
spark.sql("""
    SELECT 
        traffic_event_ts, 
        traffic_borough, 
        traffic_speed, 
        aq_pm25_ugm3, 
        weather_temperature
    FROM local.db.enriched_traffic
    ORDER BY traffic_event_ts DESC
    LIMIT 10
""").show(truncate=False)

+-------------------+---------------+-------------+------------+-------------------+
|traffic_event_ts   |traffic_borough|traffic_speed|aq_pm25_ugm3|weather_temperature|
+-------------------+---------------+-------------+------------+-------------------+
|2026-05-05 01:23:14|Staten Island  |55.3         |5.0         |63                 |
|2026-05-05 01:23:14|Staten Island  |55.3         |5.0         |64                 |
|2026-05-05 01:23:14|Staten Island  |55.3         |5.0         |63                 |
|2026-05-05 01:23:14|Staten Island  |55.3         |5.0         |59                 |
|2026-05-05 01:23:14|Staten Island  |55.3         |5.0         |64                 |
|2026-05-05 01:23:14|Staten Island  |55.3         |5.0         |64                 |
|2026-05-05 01:23:14|Staten Island  |55.3         |5.0         |63                 |
|2026-05-05 01:23:14|Staten Island  |55.3         |5.0         |63                 |
|2026-05-05 01:23:14|Staten Island  |55.3         |5.0         |5

In [3]:
# Verify the aggregations and non-null air quality readings
spark.sql("""
    SELECT 
        traffic_borough, 
        COUNT(*) as total_records,
        ROUND(AVG(traffic_speed), 2) as avg_speed_mph, 
        ROUND(AVG(aq_pm25_ugm3), 2) as avg_pm25,
        MAX(weather_temperature) as current_temp
    FROM local.db.enriched_traffic
    GROUP BY traffic_borough
    ORDER BY avg_speed_mph ASC
""").show(truncate=False)

+---------------+-------------+-------------+--------+------------+
|traffic_borough|total_records|avg_speed_mph|avg_pm25|current_temp|
+---------------+-------------+-------------+--------+------------+
|Manhattan      |4669260      |19.73        |5.2     |73          |
|Bronx          |1108830      |25.47        |NULL    |73          |
|Queens         |1867984      |28.48        |0.07    |73          |
|Staten Island  |1338370      |37.23        |4.7     |73          |
|Brooklyn       |3820742      |42.86        |3.52    |73          |
+---------------+-------------+-------------+--------+------------+



In [4]:
# 1. Read the raw Parquet dumps directly from MinIO
raw_traffic = spark.read.parquet("s3a://raw-data/traffic/")
raw_traffic.createOrReplaceTempView("raw_traffic_view")

# 2. See how many records have successfully landed
spark.sql("""
    SELECT borough, COUNT(*) as rows_ingested 
    FROM raw_traffic_view 
    GROUP BY borough
""").show()

+-------------+-------------+
|      borough|rows_ingested|
+-------------+-------------+
|       Queens|        10656|
|     Brooklyn|         3456|
|Staten Island|         7488|
|    Manhattan|         7491|
|        Bronx|         6912|
+-------------+-------------+



In [5]:
spark.read.parquet("s3a://raw-data/weather/").selectExpr(
    "MIN(kafka_timestamp) as oldest", "MAX(kafka_timestamp) as newest"
).show(truncate=False)

spark.read.parquet("s3a://raw-data/openaq/").selectExpr(
    "MIN(kafka_timestamp) as oldest", "MAX(kafka_timestamp) as newest"
).show(truncate=False)

+-----------------------+-----------------------+
|oldest                 |newest                 |
+-----------------------+-----------------------+
|2026-05-03 17:07:54.674|2026-05-05 02:08:13.217|
+-----------------------+-----------------------+

+-----------------------+----------------------+
|oldest                 |newest                |
+-----------------------+----------------------+
|2026-05-01 16:18:19.567|2026-05-05 02:09:22.68|
+-----------------------+----------------------+



In [6]:
spark.sql("""
    SELECT MIN(traffic_event_ts), MAX(traffic_event_ts), COUNT(*) as total
    FROM local.db.enriched_traffic
""").show(truncate=False)

+---------------------+---------------------+--------+
|min(traffic_event_ts)|max(traffic_event_ts)|total   |
+---------------------+---------------------+--------+
|2026-05-04 15:29:02  |2026-05-05 01:23:14  |12805186|
+---------------------+---------------------+--------+



In [7]:
spark.sql("""
    SELECT 
        MAX(traffic_event_ts) as latest_traffic,
        (SELECT MIN(kafka_timestamp) FROM parquet.`s3a://raw-data/openaq/`) as earliest_aq
    FROM local.db.enriched_traffic
""").show(truncate=False)

+-------------------+-----------------------+
|latest_traffic     |earliest_aq            |
+-------------------+-----------------------+
|2026-05-05 01:23:14|2026-05-01 16:18:19.567|
+-------------------+-----------------------+



In [8]:
spark.sql("""
    SELECT 
        traffic_borough,
        traffic_id,
        COUNT(*) as aq_matches_per_traffic_row
    FROM local.db.enriched_traffic
    GROUP BY traffic_borough, traffic_id
    ORDER BY aq_matches_per_traffic_row DESC
    LIMIT 20
""").show(truncate=False)

+---------------+----------+--------------------------+
|traffic_borough|traffic_id|aq_matches_per_traffic_row|
+---------------+----------+--------------------------+
|Manhattan      |124       |2201946                   |
|Brooklyn       |261       |1711232                   |
|Brooklyn       |258       |1657756                   |
|Manhattan      |150       |991674                    |
|Queens         |365       |832443                    |
|Manhattan      |445       |592860                    |
|Staten Island  |385       |482436                    |
|Brooklyn       |154       |154812                    |
|Bronx          |191       |97872                     |
|Bronx          |195       |97872                     |
|Bronx          |177       |95739                     |
|Manhattan      |265       |93794                     |
|Bronx          |190       |93794                     |
|Bronx          |186       |89716                     |
|Queens         |423       |89628               

In [9]:
spark.sql("""
    SELECT 
        h3_index,
        COUNT(*) as rows
    FROM local.db.enriched_traffic
    GROUP BY h3_index
    ORDER BY rows DESC
    LIMIT 10
""").show(truncate=False)

+---------------+-------+
|h3_index       |rows   |
+---------------+-------+
|882a10755dfffff|3368988|
|882a107297fffff|2201946|
|882a1072d7fffff|991674 |
|882a100d01fffff|832443 |
|882a1072c5fffff|592860 |
|882a100ae3fffff|568787 |
|882a106231fffff|482436 |
|882a100f3dfffff|214452 |
|882a10725bfffff|187588 |
|882a10729dfffff|181293 |
+---------------+-------+



In [10]:
spark.sql("""
SELECT
  MIN(traffic_event_ts) AS min_traffic,
  MAX(traffic_event_ts) AS max_traffic,
  MIN(aq_event_ts) AS min_aq,
  MAX(aq_event_ts) AS max_aq,
  MIN(weather_event_ts) AS min_weather,
  MAX(weather_event_ts) AS max_weather
FROM local.db.enriched_traffic
""").show(truncate=False)

spark.sql("""
SELECT
  COUNT(*) AS total_rows,
  SUM(CASE WHEN aq_pm25_ugm3 IS NOT NULL THEN 1 ELSE 0 END) AS aq_rows,
  SUM(CASE WHEN weather_temperature IS NOT NULL THEN 1 ELSE 0 END) AS weather_rows
FROM local.db.enriched_traffic
""").show(truncate=False)

+-------------------+-------------------+-------------------+-------------------+-------------------+-------------------+
|min_traffic        |max_traffic        |min_aq             |max_aq             |min_weather        |max_weather        |
+-------------------+-------------------+-------------------+-------------------+-------------------+-------------------+
|2026-05-04 15:29:02|2026-05-05 01:23:14|2026-05-04 16:49:38|2026-05-05 01:36:32|2026-05-04 16:42:04|2026-05-05 01:35:55|
+-------------------+-------------------+-------------------+-------------------+-------------------+-------------------+

+----------+-------+------------+
|total_rows|aq_rows|weather_rows|
+----------+-------+------------+
|12805186  |8626911|12752267    |
+----------+-------+------------+



In [11]:
# AQ Debugging

spark.sql("""
SELECT
  COUNT(*) AS total_rows,
  SUM(CASE WHEN aq_pm25_ugm3 IS NOT NULL THEN 1 ELSE 0 END) AS aq_matched_rows,
  ROUND(100.0 * SUM(CASE WHEN aq_pm25_ugm3 IS NOT NULL THEN 1 ELSE 0 END) / COUNT(*), 2) AS aq_match_pct
FROM local.db.enriched_traffic
""").show(truncate=False)

+----------+---------------+------------+
|total_rows|aq_matched_rows|aq_match_pct|
+----------+---------------+------------+
|12805186  |8626911        |67.37       |
+----------+---------------+------------+



In [12]:
spark.sql("""
SELECT
  MIN(traffic_event_ts) AS min_traffic, MAX(traffic_event_ts) AS max_traffic,
  MIN(aq_event_ts) AS min_aq, MAX(aq_event_ts) AS max_aq
FROM local.db.enriched_traffic
""").show(truncate=False)

+-------------------+-------------------+-------------------+-------------------+
|min_traffic        |max_traffic        |min_aq             |max_aq             |
+-------------------+-------------------+-------------------+-------------------+
|2026-05-04 15:29:02|2026-05-05 01:23:14|2026-05-04 16:49:38|2026-05-05 01:36:32|
+-------------------+-------------------+-------------------+-------------------+

